# Step 3 — LLM Inference Pipeline (Ollama)

This notebook loads tweets from a CSV, sends each one to a local Llama 3.2 model via Ollama,
and saves the zero-shot toxicity predictions to `/results/llm_predictions.csv`.

**Prerequisites:**
- Ollama installed and running (`ollama serve`)
- Model pulled: `ollama pull llama3.2:latest`
- A CSV in `/data/` with at least a `text` column (and optionally a `label` column)

In [ ]:
import pandas as pd
import ollama
import os
from tqdm import tqdm

# Resolve paths relative to this notebook
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
DATA_DIR     = os.path.join(PROJECT_ROOT, 'data')
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

## Configuration

In [ ]:
# ── Change these to match your actual file ──────────────────────────────────
INPUT_CSV   = os.path.join(DATA_DIR, 'tweets.csv')   # must have a 'text' column
OUTPUT_CSV  = os.path.join(RESULTS_DIR, 'llm_predictions.csv')
MODEL_NAME  = 'llama3.2:latest'
TEXT_COL    = 'text'    # column containing tweet text
LABEL_COL   = 'label'   # ground-truth column (optional; set to None if absent)
# ────────────────────────────────────────────────────────────────────────────

## Load Data

In [ ]:
df = pd.read_csv(INPUT_CSV)
print(f'Loaded {len(df):,} rows from {INPUT_CSV}')
print(df.head())

## Define Zero-Shot Prompt

In [ ]:
SYSTEM_PROMPT = (
    'You are a content moderation assistant. '
    'Your task is to classify whether a tweet is toxic or not toxic. '
    'Toxic content includes hate speech, harassment, threats, or severe insults. '
    'Respond with ONLY one word: "toxic" or "non-toxic". Do not explain.'
)

def build_user_prompt(tweet: str) -> str:
    return f'Tweet: "{tweet}"\n\nIs this tweet toxic or non-toxic?'

## Ollama Inference Helper

In [ ]:
def classify_tweet(tweet: str) -> str:
    """Send a tweet to Ollama and return the raw model response."""
    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': build_user_prompt(tweet)},
        ],
    )
    return response['message']['content'].strip().lower()


def parse_label(raw: str) -> int:
    """Convert model output to a binary label (1 = toxic, 0 = non-toxic)."""
    if 'non' in raw or raw == '0':
        return 0
    if 'toxic' in raw or raw == '1':
        return 1
    return -1  # unexpected response

## Run Inference Loop

In [ ]:
raw_responses = []
predicted_labels = []

for tweet in tqdm(df[TEXT_COL].astype(str), desc='Classifying tweets'):
    try:
        raw = classify_tweet(tweet)
    except Exception as e:
        raw = f'ERROR: {e}'
    raw_responses.append(raw)
    predicted_labels.append(parse_label(raw))

df['llm_raw_response'] = raw_responses
df['llm_predicted_label'] = predicted_labels

print('Inference complete.')
print(df[['llm_raw_response', 'llm_predicted_label']].value_counts())

## Save Predictions

In [ ]:
df.to_csv(OUTPUT_CSV, index=False)
print(f'Predictions saved to {OUTPUT_CSV}')

## Quick Evaluation (if ground-truth labels are available)

In [ ]:
if LABEL_COL and LABEL_COL in df.columns:
    from sklearn.metrics import classification_report, confusion_matrix
    import seaborn as sns
    import matplotlib.pyplot as plt

    valid = df[df['llm_predicted_label'] != -1].copy()
    y_true = valid[LABEL_COL].astype(int)
    y_pred = valid['llm_predicted_label']

    print(classification_report(y_true, y_pred, target_names=['non-toxic', 'toxic']))

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['non-toxic', 'toxic'],
                yticklabels=['non-toxic', 'toxic'], ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix — {MODEL_NAME}')
    plt.tight_layout()
    fig.savefig(os.path.join(RESULTS_DIR, 'llm_confusion_matrix.png'), dpi=150)
    plt.show()
    print('Confusion matrix saved.')
else:
    print('No ground-truth label column found — skipping evaluation.')